# Predictive task training

In [1]:
import sys
import os

project_path = "/home/sagemaker-user/gbm_hackathon"
if project_path not in sys.path:
    sys.path.append(project_path)
    print(sys.path)

['/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python310.zip', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/lib-dynload', '', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages', '/home/sagemaker-user/gbm_hackathon']


In [2]:
%load_ext autoreload
%autoreload 2
import os, sys
import pandas as pd
import numpy as np
import pickle as pkl
import torch
import seaborn as sns
import matplotlib.pyplot as plt 
import subprocess

from foundation.clinical import get_batch
from gbmhackathon.data import MosaicDataset
from gbmhackathon.s3_loader import load_s3, write_s3

In [3]:
# BUCKET_MOSAIC = "ABSTRA_DATASET_03bb30aa_16ed_4b89_913e_fe009db2aabd"
# BUCKET_PROJECT = "ABSTRA_PROJECT_STORAGE_BUCKET"

# def fetch_path(env_var_name):
#     return os.path.expandvars(f"${env_var_name}")

# S3_PATH_CLINICAL_EMB = fetch_path(BUCKET_PROJECT) + "embedding_V1/2025-03-30_14-23_clinical_emb_V1.pkl"
# S3_PATH_MODALITIES_PER_SAMPLES = fetch_path(BUCKET_MOSAIC) + "Data availibility per modality per sample.csv"
# S3_PATH_PROJECT = fetch_path(BUCKET_PROJECT)

In [4]:
# clinical_dict = load_s3(S3_PATH_CLINICAL_EMB)

In [5]:
# id2row = clinical_dict['dataset']['id2row']
# X = clinical_dict['dataset']['X']
# Y = clinical_dict['dataset']['Y']
# features = clinical_dict['dataset']['features']
# targets = clinical_dict['dataset']['targets']
# per_mod_contributions = clinical_dict['dataset']['mca_contributions']

In [6]:
# targets

In [7]:
# dict_targets = {pid:Y[id2row[pid],:] for pid in id2row.keys()}
# dict_targets

In [8]:
# freeze = False
# not freeze

In [9]:
%load_ext autoreload
%autoreload 2

from gbmhackathon.training.predictive import *
from gbmhackathon.models.mme import GBMNet
from gbmhackathon.utils.loss_functions import InfoNCELoss, RegularizedInfoNCELoss, SmoothingFunction, RankMe
from gbmhackathon.utils.module_functions import instantiate
from gbmhackathon.s3_loader import load_s3

import os
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# To investigate gradients
from torchviz import make_dot

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
torch.set_num_threads(6)
torch.get_num_threads()

6

In [11]:
device = "cpu" #"cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)

In [12]:
name_emb_dict = {"hne":"embeddings_HnE_OptimusH0.pkl",
#"spatial":"2025-03-23_18-32_spatial_emb_V1.pkl",
"clinical":"2025-03-30_14-23_clinical_emb_V1.pkl",
"wes":"2025-04-05_13-40_wes_emb_V1.pkl",
"bulk":"2025-05-03_10-15_bulk_emb_V1.pkl",
"scRNA":"2025-05-04_02-35_scRNA_emb_V1.pkl"}
pkl_storage_folder = "embedding_V1"

In [13]:
dataset = PredictiveLearningDataset(name_emb_dict, pkl_storage_folder, device=device, dropout=0.35)
print(f"Dataset size: {len(dataset)}")
BATCH_SIZE = 16
dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_predictive, generator=torch.Generator(device=dataset.device))

Using device: cpu
By keeping 35.00% of dropout augmented samples we went from:
468 dropout samples (80.41% dropout in dataset) -- to --> 178 dropout samples (60.96% dropout in dataset)
Dataset size: 292


In [14]:
toy_batch = next(iter(dataloader))
toy_batch

(['HK_G_056a_dhne_wes_bulk',
  'HK_G_002a',
  'HK_G_021a',
  'HK_G_085a_dclinical_scRNA',
  'HK_G_099a_dwes_bulk_scRNA',
  'HK_G_023a',
  'HK_G_066a',
  'HK_G_084b',
  'HK_G_001a',
  'HK_G_109b_dwes_bulk',
  'HK_G_072a_dhne_wes',
  'HK_G_070a',
  'HK_G_069a_dwes_scRNA',
  'HK_G_092b_dscRNA',
  'HK_G_086b',
  'HK_G_109b_dclinical_wes_scRNA'],
 ['hne', 'clinical', 'wes', 'bulk', 'scRNA'],
 {'hne': tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [-0.1225, -0.4985, -0.6809,  ..., -0.4195,  0.8157, -0.1495],
          [-0.3687, -0.2725, -0.5163,  ..., -0.0224,  0.6203, -0.0517],
          ...,
          [-0.1363,  0.3418,  0.6541,  ...,  0.4459,  0.2445, -0.5150],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]),
  'clinical': tensor([[ 3.5567e-01, -1.2719e+00, -1.8653e-01, -1.5341e-01, -3.6553e-01,
           -6.9805e-02,  2.8542e-01, -4.0743e-01, -1.5711e-01,  2.

In [15]:
raw_emb = toy_batch[2]
inpute_size_dict = {}
for mod in raw_emb.keys():
    print(mod, raw_emb[mod].size())
    inpute_size_dict[mod] = raw_emb[mod].size(1)

hne torch.Size([16, 1536])
clinical torch.Size([16, 12])
wes torch.Size([16, 1790])
bulk torch.Size([16, 3072])
scRNA torch.Size([16, 3072])


In [16]:
def make_patient_map(dataset, missing_mods_path: str = "s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/missing_mod_per_samples.pkl"):
    missing_mods = load_s3(missing_mods_path)
    all_ids = list(missing_mods.keys())
    patient_map = {key: i for key, i in zip(all_ids, [k for k in range(1,len(all_ids)+1)])}
    for patient_id, idx in patient_map.items():
        if patient_id.endswith('b'): # pour les rechutes on met le même index que le sample original
            patient_map[patient_id] = idx - 1
    patient_number = []
    for pid in dataset.ind2patient.values():
        if 'd' in pid:
            pid = pid[:pid.index('_d')]
        patient_number.append(patient_map[pid])
    patient_map = dict(zip(dataset.ind2patient.values(), patient_number))
    idx_compatible_mapping = dict(zip(list(set(patient_map.values())), [k for k in range(len(set(patient_map.values())))]))
    for pid in patient_map.keys():
        patient_map[pid] = idx_compatible_mapping[patient_map[pid]]
    return patient_map

In [17]:
patient_map = make_patient_map(dataset)

In [18]:
def adapt_base_config(base_cfg, input_size):
    base_copy = deepcopy(base_cfg)
    base_copy["layers"] = [input_size] + base_copy["layers"]
    return base_copy

In [19]:
out = 64
norm_fn = nn.LayerNorm
base_cfg = {"layers": [1024,512,out],
        "dropout": 0.65,
        "act_fn":SmoothingFunction,
        "norm_layer":norm_fn,
        "device":device}

hne_cfg = adapt_base_config(base_cfg, inpute_size_dict["hne"])
#spatial_cfg = adapt_base_config(small_capacity_cfg, inpute_size_dict["spatial"])
clinical_cfg = adapt_base_config(base_cfg, inpute_size_dict["clinical"])
wes_cfg = adapt_base_config(base_cfg, inpute_size_dict["wes"])
bulk_cfg = adapt_base_config(base_cfg, inpute_size_dict["bulk"])
sc_cfg = adapt_base_config(base_cfg, inpute_size_dict["scRNA"])

In [20]:
mme_bulk_cfg = {"net_type": "mlp",
            "device": device,
            "net_config": bulk_cfg}

mme_hne_cfg = {"net_type": "mlp",
            "device": device,
            "net_config": hne_cfg}

mme_sc_cfg = {"net_type": "mlp",
            "device": device,
            "net_config": sc_cfg}

mme_wes_cfg = {"net_type": "mlp",
            "device": device,
            "net_config": wes_cfg}

mme_clinical_cfg = {"net_type": "mlp",
                "device": device,
                "net_config": clinical_cfg}

In [21]:
mme_cfg = {"hne_cfg":mme_hne_cfg, 
           "clinical_cfg":mme_clinical_cfg, 
           "wes_cfg":mme_wes_cfg, 
           #"spatial_cfg":mme_spatial_cfg,
           "bulk_cfg":mme_bulk_cfg,
          "sc_cfg":mme_sc_cfg}

### For PredictionHead config: first layer depends on how many modalities are used in mme and ofinal layer is always 5 (3 regression, 1 classification (but 2 logits for classe 1 and 0)) 

In [22]:
base_head_cfg = {"layers": [out*len(mme_cfg),1024,512,5],
        "dropout": 0.65,
        "act_fn":[SmoothingFunction, SmoothingFunction, None],
        "norm_layer":norm_fn,
        "device":device}
head_cfg = {"net_type": "mlp",
            "device": device,
            "net_config": base_head_cfg}

In [23]:
gbmnet_cfg = {"head_cfg": head_cfg,
        "load_mme": False,
        "freeze_mme": False,
        "mme_path": None,
        "mme_cfg": mme_cfg}

In [24]:
gbmnet_cfg

{'head_cfg': {'net_type': 'mlp',
  'device': 'cpu',
  'net_config': {'layers': [320, 1024, 512, 5],
   'dropout': 0.65,
   'act_fn': [gbmhackathon.utils.loss_functions.SmoothingFunction,
    gbmhackathon.utils.loss_functions.SmoothingFunction,
    None],
   'norm_layer': torch.nn.modules.normalization.LayerNorm,
   'device': 'cpu'}},
 'load_mme': False,
 'freeze_mme': False,
 'mme_path': None,
 'mme_cfg': {'hne_cfg': {'net_type': 'mlp',
   'device': 'cpu',
   'net_config': {'layers': [1536, 1024, 512, 64],
    'dropout': 0.65,
    'act_fn': gbmhackathon.utils.loss_functions.SmoothingFunction,
    'norm_layer': torch.nn.modules.normalization.LayerNorm,
    'device': 'cpu'}},
  'clinical_cfg': {'net_type': 'mlp',
   'device': 'cpu',
   'net_config': {'layers': [12, 1024, 512, 64],
    'dropout': 0.65,
    'act_fn': gbmhackathon.utils.loss_functions.SmoothingFunction,
    'norm_layer': torch.nn.modules.normalization.LayerNorm,
    'device': 'cpu'}},
  'wes_cfg': {'net_type': 'mlp',
   'de

In [29]:
EPOCHS = 50
base_lr = 1e-4

gbmnet = instantiate(gbmnet_cfg, GBMNet)

optimizer = Adam(gbmnet.parameters(), lr=base_lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

mse_loss_fn = nn.MSELoss()

binary_act_fn = nn.Sigmoid()
binary_loss_fn = nn.BCELoss()
contrastive_loss_fn = RegularizedInfoNCELoss(list(name_emb_dict.keys()),
                                     patient_map, 
                                     temperature=0.05, 
                                     similarity='nt-xent', 
                                     use_all_positives=False,
                                     alpha=0.1,
                                     bound=-50,
                                     slope=0.1,
                                     rate=-5)

{'net_type': 'mlp', 'device': 'cpu', 'net_config': {'layers': [1536, 1024, 512, 64], 'dropout': 0.65, 'act_fn': <class 'gbmhackathon.utils.loss_functions.SmoothingFunction'>, 'norm_layer': <class 'torch.nn.modules.normalization.LayerNorm'>, 'device': 'cpu'}}
Using device: cpu
No potential residual connections found
Using device: cpu
No potential residual connections found
Using device: cpu
No potential residual connections found
Using device: cpu
No potential residual connections found
Using device: cpu
No potential residual connections found
Using device: cpu
No potential residual connections found


In [30]:
EPOCH_LOSSES = []
for epoch in range(EPOCHS):
    epoch_loss = []
    reg_loss_list = []
    clf_loss_list = []
    
    for idx, batch in enumerate(dataloader):
        # Get batch
        patient_ids, modalities, X_dict, avail_mods, batch_targets, targets_names = batch
        # print(batch_targets.size())
        # Forward pass
        contrastive_outputs, predictive_outputs = gbmnet(X_dict)
        # print(predictive_outputs.size())
        contrastive_loss_batch = (contrastive_outputs, patient_ids, avail_mods)

        # Loss computation (here me do contrastive and predictive at the same time)
        contrastive_loss = contrastive_loss_fn(contrastive_loss_batch)
        # print(predictive_outputs[:,:-2].size(), batch_targets[:,:-1].size())
        mse_loss = mse_loss_fn(predictive_outputs[:,:-2], batch_targets[:,:-2])
        binary_loss = binary_loss_fn(binary_act_fn(predictive_outputs[:,-2:]), batch_targets[:,-2:])
    
        loss = contrastive_loss + mse_loss + binary_loss

        # Backward pass
        loss.backward()
        optimizer.step()

        # Learning schedule
        before_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        # Store losses
        epoch_loss.append(loss.item())
        reg_loss_list.append(mse_loss.item())
        clf_loss_list.append(binary_loss.item())

    print(f"\n\nLearning Rate: {before_lr}")
    # For monitoring, not actually necessary
    pos_align = np.mean(contrastive_loss_fn.infonce.pos_alignments)
    neg_align = np.mean(contrastive_loss_fn.infonce.neg_alignments)
    print("************************************************ GLOBAL ***********************************************")
    print(f"Epoch {epoch} total loss: {np.mean(epoch_loss):.4f}".upper())
    print("*******************************************************************************************************")

    print("************************************************ EMBEDDING QUALITY ***********************************************")
    print(f"\nEpoch {epoch} Embedding quality (Alignement): {pos_align:.4f}".upper())
    print(f"Epoch {epoch} Embedding quality (Negative Alignement): {neg_align:.4f}".upper())
    print(f"Epoch {epoch} Embedding quality (Alignement ratio): {np.abs(pos_align/(neg_align + 1e-8)):.4f}".upper())
    print("*******************************************************************************************************")

    print("************************************************ PREDICTIVE POWER ************************************************")
    print(f"Epoch {epoch} MSE loss: {np.mean(reg_loss_list):.4f}".upper())
    print(f"Epoch {epoch} BCE loss: {np.mean(clf_loss_list):.4f}".upper())
    print("*******************************************************************************************************")
    contrastive_loss_fn.infonce.clear_alignments()
    EPOCH_LOSSES.append(np.mean(epoch_loss))



Learning Rate: 7.416006812042828e-05
************************************************ GLOBAL ***********************************************
EPOCH 0 TOTAL LOSS: 33.7086
*******************************************************************************************************
************************************************ EMBEDDING QUALITY ***********************************************

EPOCH 0 EMBEDDING QUALITY (ALIGNEMENT): 0.2679
EPOCH 0 EMBEDDING QUALITY (NEGATIVE ALIGNEMENT): 0.0433
EPOCH 0 EMBEDDING QUALITY (ALIGNEMENT RATIO): 6.1911
*******************************************************************************************************
************************************************ PREDICTIVE POWER ************************************************
EPOCH 0 MSE LOSS: 1.7325
EPOCH 0 BCE LOSS: 0.6389
*******************************************************************************************************
Alignments cleared


Learning Rate: 2.4195380233209022e-05
*********************

RuntimeError: all elements of input should be between 0 and 1

# There are nans popping up, we need to smooth gradient or clip them